# NB01 · Data Ingestion & Stratified Sampling

**Goal:** Load the raw 14.7 M-row CFPB complaints CSV in memory-safe chunks, profile it, resolve product-category naming conflicts, and produce a clean, balanced **20 000-row** working sample.

| Detail | Value |
|---|---|
| Environment | Local CPU
| Expected runtime | ~10–15 min (chunked I/O) |
| Output | `data/processed/sample_complaints.csv` |

## 1.1 Environment Setup

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import gc, os

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

RAW_PATH    = "../data/raw/complaints.csv"
OUTPUT_DIR  = "../data/processed"
OUTPUT_PATH = f"{OUTPUT_DIR}/sample_complaints.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE  = 50_000
TARGET_N    = 20_000
MIN_WORDS   = 20
MAX_WORDS   = 300
CREDIT_CAP  = 0.40

COL_NARRATIVE = "Consumer complaint narrative"
COL_PRODUCT   = "Product"
COL_ISSUE     = "Issue"
COL_RESPONSE  = "Company response to consumer"
COL_STATE     = "State"

MERGE_MAP = {
    "Credit reporting or other personal consumer reports":                    "Credit Reporting",
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit Reporting",
    "Credit reporting":                                                        "Credit Reporting",
    "Credit card or prepaid card":                                             "Credit Card",
    "Credit card":                                                             "Credit Card",
    "Checking or savings account":                                             "Bank Account",
    "Bank account or service":                                                 "Bank Account",
}
TOP5 = ["Credit Reporting", "Debt collection", "Credit Card", "Mortgage", "Bank Account"]

print("✓ Environment ready")
print(f"  Raw path:   {RAW_PATH}")
print(f"  Output:     {OUTPUT_PATH}")
print(f"  Chunk size: {CHUNK_SIZE:,}  |  Target N: {TARGET_N:,}")

✓ Environment ready
  Raw path:   ../data/raw/complaints.csv
  Output:     ../data/processed/sample_complaints.csv
  Chunk size: 50,000  |  Target N: 20,000


## 1.2 Chunked Dataset Profiling

A single pass through the CSV accumulates:
- total row count and per-column null counts
- word-count statistics for rows that have a narrative
- value counts for **Product** and **Issue**

In [2]:
total_rows   = 0
null_counts  = Counter()
product_ctr  = Counter()
issue_ctr    = Counter()
word_counts  = []

for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    total_rows += len(chunk)
    for col in chunk.columns:
        null_counts[col] += int(chunk[col].isna().sum())

    narr = chunk[chunk[COL_NARRATIVE].notna()]
    if len(narr):
        wc = narr[COL_NARRATIVE].str.split().str.len()
        word_counts.extend(wc.tolist())

    product_ctr.update(chunk[COL_PRODUCT].dropna().tolist())
    issue_ctr.update(chunk[COL_ISSUE].dropna().tolist())

    del chunk, narr
    gc.collect()

word_counts = np.array(word_counts)
all_columns = list(null_counts.keys())

print(f"✓ Profiling complete — {total_rows:,} rows, {len(all_columns)} columns")
print(f"  Narratives present: {len(word_counts):,}")

✓ Profiling complete — 14,704,806 rows, 18 columns
  Narratives present: 3,765,432


## 1.3 Profiling Report

In [3]:
print(f"Dataset shape: {total_rows:,} rows × {len(all_columns)} columns\n")

print(f"{'Column':<50s}  {'Nulls':>12s}  {'% Null':>7s}")
print("-" * 73)
for col in all_columns:
    n = null_counts[col]
    pct = n / total_rows * 100
    print(f"  {col:<48s}  {n:>12,}  {pct:>6.1f}%")

print(f"\nNarrative word-count percentiles (n = {len(word_counts):,}):")
for p in [5, 25, 50, 75, 95]:
    print(f"  P{p:<3d} = {int(np.percentile(word_counts, p)):>5d} words")
print(f"  Mean = {word_counts.mean():.1f}  |  Std = {word_counts.std():.1f}")

print(f"\nTop-15 Product values:")
for name, cnt in product_ctr.most_common(15):
    print(f"  {cnt:>10,}  {name}")

print(f"\nTop-15 Issue values:")
for name, cnt in issue_ctr.most_common(15):
    print(f"  {cnt:>10,}  {name}")

Dataset shape: 14,704,806 rows × 18 columns

Column                                                     Nulls   % Null
-------------------------------------------------------------------------
  Date received                                                0     0.0%
  Product                                                      0     0.0%
  Sub-product                                            235,276     1.6%
  Issue                                                        6     0.0%
  Sub-issue                                              901,285     6.1%
  Consumer complaint narrative                        10,939,374    74.4%
  Company public response                              6,724,092    45.7%
  Company                                                      0     0.0%
  State                                                   60,725     0.4%
  ZIP code                                                30,224     0.2%
  Tags                                                13,964,519   

## 1.4 Product Category Analysis & Merge Map

Several CFPB product names refer to the same logical category (e.g. three variants of *Credit reporting*). The `MERGE_MAP` defined in §1.1 collapses them. Below we verify the overlap and show the top-5 categories before and after merging.

In [4]:
print("Naming overlaps resolved by MERGE_MAP:\n")
for raw, unified in sorted(MERGE_MAP.items(), key=lambda x: x[1]):
    raw_n = product_ctr.get(raw, 0)
    print(f"  {raw_n:>10,}  {raw}")
    print(f"  {'':>10s}  ──▸  {unified}\n")

merged_ctr = Counter()
for name, cnt in product_ctr.items():
    merged_ctr[MERGE_MAP.get(name, name)] += cnt

print(f"\n{'Pre-merge top-5':<40s}  {'Post-merge top-5'}")
print("-" * 75)
pre5  = product_ctr.most_common(5)
post5 = merged_ctr.most_common(5)
for i in range(5):
    left  = f"  {pre5[i][1]:>10,}  {pre5[i][0][:30]}"
    right = f"  {post5[i][1]:>10,}  {post5[i][0][:30]}"
    print(f"{left:<40s}{right}")

Naming overlaps resolved by MERGE_MAP:

     359,020  Checking or savings account
              ──▸  Bank Account

      86,200  Bank account or service
              ──▸  Bank Account

     206,364  Credit card or prepaid card
              ──▸  Credit Card

     304,650  Credit card
              ──▸  Credit Card

   9,402,563  Credit reporting or other personal consumer reports
              ──▸  Credit Reporting

   2,163,800  Credit reporting, credit repair services, or other personal consumer reports
              ──▸  Credit Reporting

     140,426  Credit reporting
              ──▸  Credit Reporting


Pre-merge top-5                           Post-merge top-5
---------------------------------------------------------------------------
   9,402,563  Credit reporting or other pers  11,706,789  Credit Reporting
   2,163,800  Credit reporting, credit repai   1,069,518  Debt collection
   1,069,518  Debt collection                511,014  Credit Card
     446,003  Mortgage          

## 1.5 Sampling Allocation

- **Credit Reporting** is capped at 40 % of the target to prevent it from dominating.
- The remaining 60 % is distributed proportionally across the other four categories.

In [5]:
merged_narrative_ctr = Counter()
for name, cnt in product_ctr.items():
    merged_narrative_ctr[MERGE_MAP.get(name, name)] += cnt

raw_counts_all = {p: merged_narrative_ctr.get(p, 0) for p in TOP5}

alloc_preview = {}
alloc_preview["Credit Reporting"] = int(TARGET_N * CREDIT_CAP)
remaining = TARGET_N - alloc_preview["Credit Reporting"]
other_total = sum(v for k, v in raw_counts_all.items() if k != "Credit Reporting")
for p in TOP5:
    if p != "Credit Reporting":
        alloc_preview[p] = int(remaining * raw_counts_all[p] / other_total)

diff = TARGET_N - sum(alloc_preview.values())
alloc_preview["Debt collection"] += diff

print(f"{'Category':<25s}  {'Allocation':>10s}  {'%':>6s}")
print("-" * 45)
for p in TOP5:
    print(f"  {p:<23s}  {alloc_preview[p]:>10,}  {alloc_preview[p]/TARGET_N*100:>5.1f}%")
print(f"  {'TOTAL':<23s}  {sum(alloc_preview.values()):>10,}")

Category                   Allocation       %
---------------------------------------------
  Credit Reporting              8,000   40.0%
  Debt collection               5,194   26.0%
  Credit Card                   2,480   12.4%
  Mortgage                      2,165   10.8%
  Bank Account                  2,161   10.8%
  TOTAL                        20,000


## 1.6 Eligible Row Collection

Second pass through the CSV. For each chunk we keep only rows that:
1. have a non-null narrative,
2. belong to one of the **top-5 merged** categories, and
3. have a word count between 20 and 300.

In [6]:
eligible_chunks = []

for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    chunk = chunk[chunk[COL_NARRATIVE].notna()].copy()
    if chunk.empty:
        del chunk; gc.collect(); continue

    chunk["product_merged"] = chunk[COL_PRODUCT].map(MERGE_MAP).fillna(chunk[COL_PRODUCT])
    chunk = chunk[chunk["product_merged"].isin(TOP5)]
    if chunk.empty:
        del chunk; gc.collect(); continue

    chunk["word_count"] = chunk[COL_NARRATIVE].str.split().str.len()
    chunk = chunk[(chunk["word_count"] >= MIN_WORDS) & (chunk["word_count"] <= MAX_WORDS)]

    if not chunk.empty:
        keep = [COL_NARRATIVE, COL_PRODUCT, "product_merged", COL_ISSUE, COL_RESPONSE, "word_count"]
        eligible_chunks.append(chunk[keep].copy())

    del chunk; gc.collect()

df_eligible = pd.concat(eligible_chunks, ignore_index=True)
del eligible_chunks; gc.collect()

print(f"✓ Eligible rows: {len(df_eligible):,}")
for prod in TOP5:
    c = (df_eligible["product_merged"] == prod).sum()
    print(f"  {prod:<25s}  {c:>10,}")

✓ Eligible rows: 2,843,598
  Credit Reporting            2,095,484
  Debt collection               341,608
  Credit Card                   171,398
  Mortgage                       92,818
  Bank Account                  142,290


## 1.7 Stratified Sample & Validation

In [7]:
raw_counts  = {p: (df_eligible["product_merged"] == p).sum() for p in TOP5}
raw_total   = sum(raw_counts.values())

alloc = {}
alloc["Credit Reporting"] = int(TARGET_N * CREDIT_CAP)
remaining_n  = TARGET_N - alloc["Credit Reporting"]
other_total  = sum(v for k, v in raw_counts.items() if k != "Credit Reporting")
for prod in TOP5:
    if prod != "Credit Reporting":
        alloc[prod] = int(remaining_n * raw_counts[prod] / other_total)

diff = TARGET_N - sum(alloc.values())
alloc["Debt collection"] += diff

print(f"{'Product':<25}  {'Target':>7}  {'Pool':>10}  {'%':>6}")
print("-" * 58)
for p in TOP5:
    print(f"  {p:<23}  {alloc[p]:>7,}  {raw_counts[p]:>10,}  {alloc[p]/TARGET_N*100:>5.1f}%")
print(f"  {'TOTAL':<23}  {sum(alloc.values()):>7,}")

parts = []
for prod, n in alloc.items():
    pool = df_eligible[df_eligible["product_merged"] == prod]
    parts.append(pool.sample(n=n, random_state=RANDOM_STATE))
    print(f"  Sampled {n:,} from {len(pool):,} eligible  [{prod}]")

df_sample = (pd.concat(parts, ignore_index=True)
               .sample(frac=1, random_state=RANDOM_STATE)
               .reset_index(drop=True))

assert len(df_sample) == TARGET_N, f"Expected {TARGET_N}, got {len(df_sample)}"
print(f"\n✓ Final sample: {len(df_sample):,} rows")

Product                     Target        Pool       %
----------------------------------------------------------
  Credit Reporting           8,000   2,095,484   40.0%
  Debt collection            5,481     341,608   27.4%
  Credit Card                2,749     171,398   13.7%
  Mortgage                   1,488      92,818    7.4%
  Bank Account               2,282     142,290   11.4%
  TOTAL                     20,000
  Sampled 8,000 from 2,095,484 eligible  [Credit Reporting]
  Sampled 5,481 from 341,608 eligible  [Debt collection]
  Sampled 2,749 from 171,398 eligible  [Credit Card]
  Sampled 1,488 from 92,818 eligible  [Mortgage]
  Sampled 2,282 from 142,290 eligible  [Bank Account]

✓ Final sample: 20,000 rows


## 1.8 Save

Rename to analysis-friendly column names, keep only the columns needed downstream, and write to disk.

In [9]:
df_out = df_sample.rename(columns={
    COL_NARRATIVE: "narrative",
    COL_PRODUCT:   "product_raw",
    "product_merged": "product",
    COL_ISSUE:     "issue",
    COL_RESPONSE:  "company_response",
})

df_out = df_out[["narrative", "product", "product_raw", "issue", "company_response", "word_count"]]

df_out.to_csv(OUTPUT_PATH, index=False)
print(f"✓ Saved {len(df_out):,} rows → {OUTPUT_PATH}")
print(f"  File size: {os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB")

✓ Saved 20,000 rows → ../data/processed/sample_complaints.csv
  File size: 16.3 MB
